In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [5]:
df_train = pd.read_csv('./dataset/train.csv')
df_train.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [6]:
df_test = pd.read_csv('./dataset/test.csv')
df_test.head()

,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,49,2:15,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213,Snowy
2,2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.318203,Sunny
3,3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,NaN,Rainy
4,4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.266162,Foggy


In [7]:
print(df_train.shape)
print(df_test.shape)

(77299, 11)
(41778, 10)


In [8]:
categorical_cols = df_train.select_dtypes(include=['object']).columns
print(categorical_cols.tolist())

['geohash', 'timestamp', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']


In [9]:
df_train['day'].unique()

array([48, 49])

In [10]:
df_train['geohash'].value_counts()

geohash
qp03wd    105
qp03wf    105
qp09t0    105
qp03w9    105
qp03x3    105
         ... 
qp08gs      1
qp08fq      1
qp0d1t      1
qp09vc      1
qp09jc      1
Name: count, Length: 1249, dtype: int64

In [11]:
print("="*60)
print("1. BASIC SHAPE & DATA TYPES")
print("="*60)
print(f"Train Shape: {df_train.shape}")
print(f"Test Shape:  {df_test.shape}\n")
print(df_train.info())

1. BASIC SHAPE & DATA TYPES
Train Shape: (77299, 11)
Test Shape:  (41778, 10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  object 
 4   demand         77299 non-null  float64
 5   RoadType       76699 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    74804 non-null  float64
 10  Weather        76502 non-null  object 
dtypes: float64(2), int64(3), object(6)
memory usage: 6.5+ MB
None


In [13]:
print("\n" + "="*60)
print("2. TARGET VARIABLE (demand) DEEP DIVE")
print("="*60)
demand_stats = df_train['demand'].describe()
skewness = df_train['demand'].skew()
kurtosis = df_train['demand'].kurtosis()
zeros_count = (df_train['demand'] == 0).sum()
print(demand_stats)
print(f"Skewness: {skewness:.2f} (If > 1, highly skewed. Consider log1p transform)")
print(f"Kurtosis: {kurtosis:.2f} (High values mean heavy tails/extreme outliers)")
print(f"Number of exact 0 demand rows: {zeros_count} ({zeros_count/len(df_train)*100:.2f}%)")


2. TARGET VARIABLE (demand) DEEP DIVE
count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64
Skewness: 3.73 (If > 1, highly skewed. Consider log1p transform)
Kurtosis: 17.33 (High values mean heavy tails/extreme outliers)
Number of exact 0 demand rows: 0 (0.00%)


In [14]:
print("\n" + "="*60)
print("3. MISSING VALUE PATTERNS (TRAIN vs TEST)")
print("="*60)
missing_train = df_train.isnull().sum()
missing_test = df_test.isnull().sum()
missing_df = pd.DataFrame({
    'Train Missing': missing_train, 
    'Train %': (missing_train/len(df_train))*100,
    'Test Missing': missing_test,
    'Test %': (missing_test/len(df_test))*100
})
print(missing_df[missing_df['Train Missing'] > 0])

print("\n" + "="*60)


3. MISSING VALUE PATTERNS (TRAIN vs TEST)
             Train Missing   Train %  Test Missing    Test %
RoadType               600  0.776207         324.0  0.775528
Temperature           2495  3.227726        1349.0  3.228972
Weather                797  1.031061         431.0  1.031643



In [15]:
print("4. ARE MISSING VALUES LINKED TO SPECIFIC DAYS/TIMES?")
print("="*60)
# Let's check if weather or temperature is missing completely at random or on a specific day
missing_by_day = df_train.groupby('day')[['Weather', 'Temperature', 'RoadType']].apply(lambda x: x.isnull().sum())
print("Missing counts grouped by Day:")
print(missing_by_day)

print("\n" + "="*60)

4. ARE MISSING VALUES LINKED TO SPECIFIC DAYS/TIMES?
Missing counts grouped by Day:
     Weather  Temperature  RoadType
day                                
48       716         2241       539
49        81          254        61



In [16]:
print("5. GEOHASH (SPATIAL) ALIGNMENT BETWEEN TRAIN & TEST")
print("="*60)
train_geohashes = set(df_train['geohash'].unique())
test_geohashes = set(df_test['geohash'].unique())
unseen_test_geo = test_geohashes - train_geohashes

print(f"Unique Geohashes in Train: {len(train_geohashes)}")
print(f"Unique Geohashes in Test:  {len(test_geohashes)}")
print(f"Geohashes in Test but NOT in Train: {len(unseen_test_geo)}")
if len(unseen_test_geo) > 0:
    print("⚠️ WARNING: Cold-start problem! We have new locations in test set.")
else:
    print("✅ Great news: All test locations exist in the training set.")

5. GEOHASH (SPATIAL) ALIGNMENT BETWEEN TRAIN & TEST
Unique Geohashes in Train: 1249
Unique Geohashes in Test:  1190
Geohashes in Test but NOT in Train: 10
⚠️ WARNING: Cold-start problem! We have new locations in test set.


In [19]:
# You may need to run: !pip install pygeohash
import pygeohash as pgh
import numpy as np

print("="*60)
print("1. DECODING GEOHASHES TO LATITUDE & LONGITUDE")
print("="*60)

# Decode geohash to exact coordinates
df_train['latitude'] = df_train['geohash'].apply(lambda x: pgh.decode(x)[0])
df_train['longitude'] = df_train['geohash'].apply(lambda x: pgh.decode(x)[1])
df_test['latitude'] = df_test['geohash'].apply(lambda x: pgh.decode(x)[0])
df_test['longitude'] = df_test['geohash'].apply(lambda x: pgh.decode(x)[1])

print(f"Latitude Range:  {df_train['latitude'].min():.4f} to {df_train['latitude'].max():.4f}")
print(f"Longitude Range: {df_train['longitude'].min():.4f} to {df_train['longitude'].max():.4f}")

print("\n" + "="*60)
print("2. ENGINEERING CYCLICAL TIME FEATURES")
print("="*60)

def extract_time_features(df):
    # Split H:M into integers
    df[['hour', 'minute']] = df['timestamp'].str.split(':', expand=True).astype(int)
    
    # Convert to total minutes from midnight (0 to 1439)
    df['total_minutes'] = df['hour'] * 60 + df['minute']
    
    # Cyclical Encoding for Time
    # 1440 minutes in a day. We use sine and cosine to map this to a circle.
    df['time_sin'] = np.sin(2 * np.pi * df['total_minutes'] / 1440)
    df['time_cos'] = np.cos(2 * np.pi * df['total_minutes'] / 1440)
    
    # Drop intermediate columns if desired, but keep total_minutes as it's useful for trees
    return df

df_train = extract_time_features(df_train)
df_test = extract_time_features(df_test)

print("Time features created successfully.")
print(df_train[['timestamp', 'hour', 'minute', 'total_minutes', 'time_sin', 'time_cos']].head())

1. DECODING GEOHASHES TO LATITUDE & LONGITUDE
Latitude Range:  -5.4849 to -5.2377
Longitude Range: 90.5878 to 90.9723

2. ENGINEERING CYCLICAL TIME FEATURES
Time features created successfully.
  timestamp  hour  minute  total_minutes  time_sin  time_cos
0       0:0     0       0              0       0.0       1.0
1       0:0     0       0              0       0.0       1.0
2       0:0     0       0              0       0.0       1.0
3       0:0     0       0              0       0.0       1.0
4       0:0     0       0              0       0.0       1.0


In [20]:
import numpy as np

print("="*60)
print("1. SMART IMPUTATION (FIXING SENSOR FAILURES)")
print("="*60)

def impute_missing_data(df):
    # 1. Sort temporally and spatially so ffill makes logical sense
    df = df.sort_values(by=['geohash', 'day', 'total_minutes']).reset_index(drop=True)
    
    # 2. Forward fill then backward fill continuous/weather data within each specific location
    df['Temperature'] = df.groupby('geohash')['Temperature'].ffill().bfill()
    df['Weather'] = df.groupby('geohash')['Weather'].ffill().bfill()
    
    # 3. For RoadType, fill missing values with the most common road type (mode) for that exact geohash
    df['RoadType'] = df.groupby('geohash')['RoadType'].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown')
    )
    
    return df

df_train = impute_missing_data(df_train)
df_test = impute_missing_data(df_test)

print("Remaining Missing Values in Train:")
print(df_train[['Temperature', 'Weather', 'RoadType']].isnull().sum())


print("\n" + "="*60)
print("2. TARGET TRANSFORMATION (FIXING SKEWNESS)")
print("="*60)

# Apply log1p transformation to bring extreme demand spikes closer to the median
df_train['demand_log'] = np.log1p(df_train['demand'])

print(f"Original Target Skewness: {df_train['demand'].skew():.4f}")
print(f"New Transformed Skewness: {df_train['demand_log'].skew():.4f}")
print("✅ Target normalized. (Remember to use np.expm1() on your final predictions!)")

1. SMART IMPUTATION (FIXING SENSOR FAILURES)
Remaining Missing Values in Train:
Temperature    0
Weather        0
RoadType       0
dtype: int64

2. TARGET TRANSFORMATION (FIXING SKEWNESS)
Original Target Skewness: 3.7285
New Transformed Skewness: 2.9651
✅ Target normalized. (Remember to use np.expm1() on your final predictions!)


In [21]:
import lightgbm as lgb
from sklearn.metrics import r2_score

print("="*60)
print("1. ENCODING CATEGORICAL FEATURES FOR TREE MODELS")
print("="*60)

# Identify features that are text/categorical
categorical_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

# Convert them to pandas 'category' type which LightGBM/CatBoost understand natively
for col in categorical_cols:
    df_train[col] = df_train[col].astype('category')
    df_test[col] = df_test[col].astype('category')

# Define exactly which columns the model will use to learn
features = [
    'latitude', 'longitude', 'hour', 'minute', 'total_minutes', 
    'time_sin', 'time_cos', 'NumberofLanes',
    'RoadType', 'LargeVehicles', 'Landmarks', 'Weather'
]

print("Features selected for training:")
print(features)

print("\n" + "="*60)
print("2. TIME-BASED VALIDATION SPLIT (TRAIN: DAY 48 | VAL: DAY 49)")
print("="*60)

# Split based on the 'day' column to prevent future-data leakage
X_train = df_train[df_train['day'] == 48][features]
y_train = df_train[df_train['day'] == 48]['demand_log']  # Target is the log version

X_val = df_train[df_train['day'] == 49][features]
y_val = df_train[df_train['day'] == 49]['demand_log']

print(f"Train Set (Day 48) Shape:      {X_train.shape}")
print(f"Validation Set (Day 49) Shape: {X_val.shape}")

# Double check if any missing values crept into our features
assert X_train.isnull().sum().sum() == 0, "Wait, there are nulls in X_train!"
assert X_val.isnull().sum().sum() == 0, "Wait, there are nulls in X_val!"
print("✅ Split successful! No missing data leaked.")

1. ENCODING CATEGORICAL FEATURES FOR TREE MODELS
Features selected for training:
['latitude', 'longitude', 'hour', 'minute', 'total_minutes', 'time_sin', 'time_cos', 'NumberofLanes', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

2. TIME-BASED VALIDATION SPLIT (TRAIN: DAY 48 | VAL: DAY 49)
Train Set (Day 48) Shape:      (69427, 12)
Validation Set (Day 49) Shape: (7872, 12)
✅ Split successful! No missing data leaked.


In [22]:
import lightgbm as lgb
import numpy as np
from sklearn.metrics import r2_score

print("="*60)
print("1. TRAINING LIGHTGBM REGRESSOR")
print("="*60)

# Define basic hyper-parameters suitable for this scale of data
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'random_state': 42,
    'n_jobs': -1
}

model = lgb.LGBMRegressor(**params)

# Train the model with early stopping to avoid overfitting
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

print("✅ Model training complete!")

print("\n" + "="*60)
print("2. EVALUATING LEADERBOARD METRIC (R2 SCORE)")
print("="*60)

# Get predictions on the log scale
val_preds_log = model.predict(X_val)

# Convert predictions AND actual values back to the original demand scale
val_preds_original = np.expm1(val_preds_log)
y_val_original = np.expm1(y_val)

# Calculate R2 Score
r2 = r2_score(y_val_original, val_preds_original)
print(f"Validation R2 Score (Original Scale): {r2:.5f}")

print("\n" + "="*60)
print("3. FEATURE IMPORTANCE (WHAT DROVE THE PREDICTIONS?)")
print("="*60)

# Check which features the model relied on the most
importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importance)

1. TRAINING LIGHTGBM REGRESSOR
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000779 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 376
[LightGBM] [Info] Number of data points in the train set: 69427, number of used features: 12
[LightGBM] [Info] Start training from score 0.082019
✅ Model training complete!

2. EVALUATING LEADERBOARD METRIC (R2 SCORE)
Validation R2 Score (Original Scale): 0.76579

3. FEATURE IMPORTANCE (WHAT DROVE THE PREDICTIONS?)
          Feature  Importance
0        latitude        2432
1       longitude        2032
4   total_minutes         445
8        RoadType         311
5        time_sin         269
6        time_cos         249
2            hour         143
9   LargeVehicles          72
7   NumberofLanes          40
3          minute           5
10      Landmarks           2
11        Weather           0
